In [0]:
# Lab 1 / Phase 1 / Stage C — UC Function tool: cash flow summary by company code
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_cashflow_summary(
    p_company_code STRING COMMENT 'SAP company code, e.g. 1010 or 1710'
)
RETURNS TABLE (
    company_code      STRING,
    currency          STRING,
    months_covered    BIGINT,
    total_net_cashflow DECIMAL(38,4),
    total_inflow      DECIMAL(38,4),
    total_outflow     DECIMAL(38,4),
    avg_monthly_net   DECIMAL(38,4)
)
COMMENT 'Returns the overall cash flow summary for a given SAP company code: total net cash flow, total inflow, total outflow, and average monthly net, aggregated across all months. Use this to answer questions about a specific company code cash flow.'
RETURN
    SELECT
        company_code,
        MAX(currency)                  AS currency,
        COUNT(*)                       AS months_covered,
        SUM(net_cashflow)              AS total_net_cashflow,
        SUM(total_inflow)              AS total_inflow,
        SUM(total_outflow)             AS total_outflow,
        ROUND(AVG(net_cashflow),2)     AS avg_monthly_net
    FROM workspace.default.gold_monthly_cashflow
    WHERE company_code = p_company_code
    GROUP BY company_code
""")

print("Test — company code 1010:")
spark.sql("SELECT * FROM workspace.default.get_cashflow_summary('1010')").show()
print("Test — company code 1710:")
spark.sql("SELECT * FROM workspace.default.get_cashflow_summary('1710')").show()